In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
from statsmodels.stats.multitest import multipletests
import time

# paths
AA_GENO  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports done. Output dir:", OUT_DIR)

Imports done. Output dir: C:\Users\user\Desktop\ai causal\FTND\african_amercian


In [2]:
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

# filter to smokers only
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["ftnd"] = pd.to_numeric(smokers["ftnd"], errors="coerce")

# drop any remaining missing FTND
smokers = smokers.dropna(subset=["ftnd"])
smokers = smokers[smokers["ftnd"] >= 0]  # remove any -9 codes

print("Total AA samples:", len(meta_df))
print("Smokers with valid FTND:", len(smokers))
print("FTND distribution:")
print(smokers["ftnd"].describe())
print("FTND value counts:")
print(smokers["ftnd"].value_counts().sort_index())

smoker_ids = smokers["sample_id"].tolist()

Total AA samples: 3348
Smokers with valid FTND: 1631
FTND distribution:
count    1631.000000
mean        8.429185
std         1.481495
min         0.000000
25%         9.000000
50%         9.000000
75%         9.000000
max        10.000000
Name: ftnd, dtype: float64
FTND value counts:
ftnd
0        4
1        4
2        9
3       18
4       28
5       38
6       57
7       99
8      127
9     1151
10      96
Name: count, dtype: int64


In [3]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# load manifest for chromosome info
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)

# load genotype matrix
encoded_df = pd.read_csv(AA_GENO)
probe_id_array = encoded_df["probe_id"].to_numpy()
all_sample_ids = encoded_df.columns[1:].tolist()

# filter columns to smokers only
smoker_id_set = set(smoker_ids)
smoker_cols = [sid for sid in all_sample_ids if sid in smoker_id_set]
print("Smoker columns found in genotype file:", len(smoker_cols))

# align smokers metadata to genotype column order
smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols).reset_index()
Y_ftnd = smokers_aligned["ftnd"].values.astype(np.float64)
age_std = ((smokers_aligned["age"].astype(float) - smokers_aligned["age"].astype(float).mean())
           / smokers_aligned["age"].astype(float).std()).values
gender_binary = (smokers_aligned["gender"] == "Male").astype(np.float64).values

print("Y_ftnd shape:", Y_ftnd.shape)
print("Y_ftnd mean:", Y_ftnd.mean().round(3))

# filter to autosomal probes
keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])

# extract smoker columns as numpy array
X_snp = encoded_df[smoker_cols].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T  # smokers x SNPs
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp
gc.collect()

print("X_auto shape:", X_auto.shape)  # expect (1631, 233610)

Smoker columns found in genotype file: 1631
Y_ftnd shape: (1631,)
Y_ftnd mean: 8.429
X_auto shape: (1631, 233610)


In [4]:
# EIGENSTRAT standardization
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Monomorphic excluded:", (~valid_mask).sum())
print("Informative retained:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]

del X_float
gc.collect()

print("X_std shape:", X_std.shape)

# PCA top 10
pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# confounder matrix: [PC1..10, age, gender]
X_conf = np.hstack([
    pcs,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("X_conf shape:", X_conf.shape)  # expect (1631, 12)

Monomorphic excluded: 106146
Informative retained: 127464
X_std shape: (1631, 127464)
Explained variance ratio: [0.00717025 0.00188095 0.00178024 0.00172667 0.00171314 0.0015741
 0.0015012  0.00145438 0.0014028  0.00138412]
X_conf shape: (1631, 12)


In [6]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Running single test iteration...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.01, 0.001, 0.0001]:
    print(f"Raw p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

Running single test iteration...
Completed in 34.6s
Raw p < 0.01: 3034 SNPs
Raw p < 0.001: 1267 SNPs
Raw p < 0.0001: 512 SNPs


In [7]:
n_repeats = 30
threshold = 0.0001
n_snps = X_std.shape[1]

significant_counts = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction = significant_counts / n_repeats

stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df.to_csv(os.path.join(OUT_DIR, "ftnd_stability_results.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction >= thresh).sum()} SNPs")

Running 30 stability repeats (p<0.0001)...
  Repeat 5/30 — 2.8 min
  Repeat 10/30 — 5.6 min
  Repeat 15/30 — 8.3 min
  Repeat 20/30 — 10.9 min
  Repeat 25/30 — 13.6 min
  Repeat 30/30 — 16.3 min

Total: 16.3 min
  >= 50% stability: 532 SNPs
  >= 70% stability: 511 SNPs
  >= 80% stability: 503 SNPs
  >= 90% stability: 491 SNPs
  >= 100% stability: 428 SNPs


In [8]:
# check if we should use stricter threshold
print("Current (p<0.0001, 100% stability): 428 SNPs")
print("\nLet's check what p<0.00001 gives in one iteration:")

start = time.time()
_, pvals_strict = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.00001, 0.000001, 0.0000001]:
    print(f"Raw p < {thresh}: {(pvals_strict < thresh).sum()} SNPs")

# also check top SNPs by stability fraction
stability_df = pd.read_csv(os.path.join(OUT_DIR, "ftnd_stability_results.csv"))
print("\nTop 20 most stable SNPs:")
print(stability_df.head(20)[["probe_id", "stability_fraction", "n_significant_repeats"]])

Current (p<0.0001, 100% stability): 428 SNPs

Let's check what p<0.00001 gives in one iteration:
Completed in 24.2s
Raw p < 1e-05: 285 SNPs
Raw p < 1e-06: 158 SNPs
Raw p < 1e-07: 91 SNPs

Top 20 most stable SNPs:
                       probe_id  stability_fraction  n_significant_repeats
0     exm26794-0_B_R_1921571564                 1.0                     30
1    exm326427-0_B_R_1922217674                 1.0                     30
2    exm223412-0_B_F_1918813765                 1.0                     30
3    exm523594-0_T_F_1921962613                 1.0                     30
4    exm551051-0_T_F_1921880849                 1.0                     30
5    exm294694-0_B_F_1922223929                 1.0                     30
6    exm466812-0_T_R_1921194521                 1.0                     30
7   exm1189236-0_B_R_1922835354                 1.0                     30
8   exm1189432-0_T_R_1922833074                 1.0                     30
9   exm1191937-0_B_F_1922855101      

In [ ]:
n_repeats = 30
threshold = 1e-6
n_snps = X_std.shape[1]

significant_counts_v2 = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=rep)
    significant_counts_v2 += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction_v2 = significant_counts_v2 / n_repeats

stability_df_v2 = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction_v2,
    "n_significant_repeats": significant_counts_v2
}).sort_values("stability_fraction", ascending=False)

stability_df_v2.to_csv(os.path.join(OUT_DIR, "ftnd_stability_results_v2.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction_v2 >= thresh).sum()} SNPs")

Running 30 stability repeats (p<1e-06)...
  Repeat 5/30 — 2.8 min
  Repeat 10/30 — 5.7 min
  Repeat 15/30 — 8.5 min
